# Introduction: Real-Time Communication using WebSockets
In this module, we’ll learn how WebSockets enable persistent, bidirectional communication between clients and servers. By the end of this module, you will be able to explain and implement the following concepts:

* **Describe the WebSocket architecture**, including the HTTP-based handshake process that upgrades a connection from the traditional request model underlying modern Web services to a frame-based channel.
* **Parse and construct WebSocket frames**: Understanding the header fields (FIN, opcode, mask, length, etc.) and how text vs. binary payloads are handled is crucial to mastering WebSockets. Your communication library typically handles these low-level details, but it's still good to know!

As applications demand ever-lower latency and instantaneous updates, WebSockets provide a single, long-lived TCP socket over which both the client and server can push messages at any time, making them ideal for chat, live dashboards, multiplayer games, and real-time trading platforms.

Let's get started!
___

## Why a WebSocket?
WebSockets bridge the gap between the simple HTTP request/response model and the needs of modern real-time applications. 
* __Persistence__: Your client and server maintain a constant, two-way conversation, allowing messages to flow instantly over a persistent connection. Thus, WebSockets eliminate the need for repeated HTTP requests and responses (called polling), which can introduce latency and overhead, and can miss data updates.
* __Lower latency & overhead__: After an initial HTTP-based _handshake_, a single TCP connection is kept open. There’s no per-message HTTP header exchange, so payloads are smaller and round‐trip delays drop. Further, WebSockets let the client and server _push_ updates onto the connection the moment they happen (which is unlike traditional HTTP, where the client must request data from the server).
* __Full-duplex channel__: Both client and server can send independently at any time. This makes WebSockets ideal for chat apps, live dashboards, multiplayer games, IoT telemetry, and collaborative editing, among other applications.

WebSockets are cool because they enable real-time two-way communication. Let's dig a _little_ deeper into how WebSockets work.

## Handshake
Every client, e.g., a browser, a mobile app, or an IoT device, must first perform a quick, HTTP-based _handshake_. Both sides of the conversation must agree to switch from the traditional request/response model to a long-lived, bidirectional TCP connection. Here’s how it works:

* **Client asks to upgrade.** Your application sends a standard HTTP `GET` request to the server, but adds special headers indicating “I’d like to switch to WebSockets,” ensuring the underlying TCP connection between the client and server stays open.
* **Server verifies and confirms.** The server checks a _nonce_ to prevent spoofing, then replies with `101 Switching Protocols` and a matching token to say “OK, let’s do this.”
* **Protocol switch.**. Once confirmed, both sides stop using the request/response model and exchanging HTTP messages and instead begin exchanging compact WebSocket _frames_ over the same TCP connection.
* **Persistent, two-way channel.** The TCP socket remains open indefinitely (until closed), allowing your client app and server to send messages to each other at any time with minimal framing overhead. On the server side, new data can be pushed to the client as soon as it’s available, without waiting for a request. Cool!

### What is a nonce?
A **nonce** (short for “number used once”) is a random, single-use value that helps ensure each handshake or transaction is unique and can’t be replayed. Think of it as a temporary password that’s valid only for the current session. Here’s why nonces are necessary in WebSocket handshakes:

* **Uniqueness**. Every time you initiate a WebSocket (or any secure) handshake, you generate a fresh nonce so the server can tell this is a brand-new request. Every nonce is unique to that specific handshake.
* **Proof against replay**. Because the server signs or echoes back the nonce (after hashing), an attacker can’t capture an old handshake and replay it to trick your server. This ensures that even if someone intercepts the handshake, they can’t reuse it later, e.g., to hijack a session. Thus, using a nonce adds minimal overhead but has a significant security benefit.

### What is 101 Switching Protocols?
The HTTP `101 Switching Protocols` code is an HTTP/1.1 status code the server sends back to say, “Okay, I accept your request to switch from HTTP to the new protocol (e.g., WebSocket).”

* **Status 1xx (Informational)**: The 1xx set of status codes doesn’t carry a page or any data; they just let the client know that the server is processing the request and has accepted the protocol switch.
* **“Switching Protocols”**: Echoes the client’s desire to upgrade, confirming both sides will stop speaking HTTP and start using WebSocket frames.
* **Stays on the same TCP socket**: Finally, no new connection is opened; all communication will happen on the existing socket, now running the agreed-upon protocol.

Ok, so let's assume the handshake is successful. What happens next? Well, the client and server can now exchange messages using WebSocket frames. These frames are significantly lighter than traditional HTTP requests, enabling faster and more efficient communication. 

Let's take a look at how these frames work.
___

## Frames
Once the handshake completes, client and server exchange data in **WebSocket frames**, which are self-contained packets that carry both metadata (stored in a compact header) and your actual message (the payload). Below is the header format for each frame:

| Field            | Size             | Description                                                                                                                                                          |
| ---------------- | ---------------- | -------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **FIN + RSV1–3** | 4 bits           | **FIN:** set to 1 on the final frame of a message. <br> **RSV1–3:** reserved for negotiated extensions (e.g. compression). Usually 0.                                |
| **Opcode**       | 4 bits           | Type of frame:<br>• 0 = continuation frame<br>• 1 = text data<br>• 2 = binary data<br>• 8 = connection close<br>• 9 = ping<br>• 10 = pong                            |
| **Mask bit**     | 1 bit            | Client→server frames **must** set this to 1 (so the server knows payload is masked). Server→client frames **must** set to 0.                                         |
| **Payload len**  | 7 bits (+ ext)   | If ≤ 125, that value is the payload length in bytes. <br> If 126, the next 2 bytes (16-bit) give the length. <br> If 127, the next 8 bytes (64-bit) give the length. |
| **Extended len** | 0, 2, or 8 bytes | Present only when the 7-bit length is 126 or 127, to hold the full payload length.                                                                                   |
| **Masking key**  | 0 or 4 bytes     | A 4-byte random key used to unmask the payload. Present **only** when Mask bit = 1.                                                                                  |
| **Payload data** | (variable)       | The actual application data. If masked, each byte is XOR’d with the masking key.                                                                                     |

---

Wow! That's a lot of information! But don't worry, you don't need to memorize all these details. 
* __In practice__: your client application library will handle framing for you. Your job is just to send or receive “messages.” But under the hood, every send/receive corresponds to one or more of these frames, providing the reliability, control, and efficiency that power real-time WebSocket apps.
* __Why frames?__: They allow WebSockets to be lightweight and efficient. Instead of sending large HTTP headers with every message, WebSocket frames have a compact header that carries just enough metadata to route and process the message. This reduces overhead and improves performance, especially for high-frequency updates.

In Julia, you can use [the `HTTP.jl` package](https://github.com/JuliaWeb/HTTP.jl) to work with WebSockets. The package provides a simple interface for creating WebSocket clients and servers, handling the framing and message parsing for you. 

Coincidentally, the `HTTP.jl` package is the same package that we used when building our RESTful API in the previous module. Nice! Always buy, never build!
___